# Experiment: Visual inspection of training inputs

Objective:
- Inspect the exact tensors consumed by the model during training (`u, v, w, u_mag, v_mag, w_mag`).
- Validate shapes, ranges, normalization, and patch sampling behavior from the NIfTI pipeline.
- Run one model inference pass and verify 4-channel output (`u, v, w, mag`).


In [ ]:
from __future__ import annotations

import csv
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Could not find repository root (.git).")


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

SEED = 7
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SRC_DIR = {SRC_DIR}")
SEED


## Training DataLoader Configuration

Set these to match your `src/trainer_nifti.py` run. This notebook defaults to `predict_mag=True` (4-channel training target).


In [ ]:
TRAIN_CSV = PROJECT_ROOT / "data/paired_dataset/train_random_1.csv"

PATCH_SIZE = 16
RES_INCREASE = 2
BATCH_SIZE = 2
TRAIN_SAMPLES_PER_VOLUME = 32

MAG_SCALE = 4095.0
MASK_THRESHOLD = 0.5
RAW_PHASE_INPUT = True
LEGACY_INVERT_UV_SIGN_ON_RAW = False

NO_AUGMENTATION = False
DETERMINISTIC_TRAIN_PATCHES = False
ROTATION_PROB = 0.5

PREDICT_MAG = True
CACHE_DATASET = False
CACHE_EAGER = True
NUM_WORKERS = 0

SAMPLE_INDEX = 0
assert TRAIN_CSV.exists(), f"Training CSV not found: {TRAIN_CSV}"
TRAIN_CSV


In [ ]:
with TRAIN_CSV.open("r", newline="") as f:
    reader = csv.DictReader(f)
    first_row = next(reader)

print("CSV columns:")
print(reader.fieldnames)
print("\nFirst row (main fields):")
for key in ["lr_u", "lr_v", "lr_w", "lr_mag_u", "hr_u", "hr_v", "hr_w", "hr_mag", "mask", "venc", "time_start", "time_end", "time_index"]:
    if key in first_row:
        print(f"- {key}: {first_row.get(key)}")


In [ ]:
from Network.NiftiPatchDataset import create_nifti_patch_dataloader

train_rotation_prob = 0.0 if NO_AUGMENTATION else max(float(ROTATION_PROB), 0.0)
train_random_time_frame = not NO_AUGMENTATION
train_random_patch = not DETERMINISTIC_TRAIN_PATCHES

train_loader = create_nifti_patch_dataloader(
    csv_path=str(TRAIN_CSV),
    patch_size=PATCH_SIZE,
    res_increase=RES_INCREASE,
    batch_size=BATCH_SIZE,
    samples_per_volume=TRAIN_SAMPLES_PER_VOLUME,
    shuffle=True,
    augment=True,
    include_hr_mag=PREDICT_MAG,
    cache_dataset=CACHE_DATASET,
    cache_eager=CACHE_EAGER,
    random_time_frame=train_random_time_frame,
    random_patch_sampling=train_random_patch,
    rotation_prob=train_rotation_prob,
    num_workers=NUM_WORKERS,
    mag_scale=MAG_SCALE,
    mask_threshold=MASK_THRESHOLD,
    raw_phase_input=RAW_PHASE_INPUT,
    invert_uv_sign_on_raw=LEGACY_INVERT_UV_SIGN_ON_RAW,
    minimum_coverage=0.0,
    max_sampling_attempts=100,
    allow_empty_fallback=True,
)

batch = next(iter(train_loader))
print(f"Number of tensors in batch: {len(batch)}")
print(f"Batches per epoch (len(loader)): {len(train_loader)}")


In [ ]:
if len(batch) == 11:
    u, v, w, u_mag, v_mag, w_mag, u_hr, v_hr, w_hr, venc, mask = batch
    hr_mag = None
elif len(batch) == 12:
    u, v, w, u_mag, v_mag, w_mag, u_hr, v_hr, w_hr, hr_mag, venc, mask = batch
else:
    raise ValueError(f"Unexpected number of tensors in batch: {len(batch)}")

if PREDICT_MAG:
    assert hr_mag is not None, "predict_mag=True but hr_mag is missing in the batch."

tensor_map = {
    "u": u,
    "v": v,
    "w": w,
    "u_mag": u_mag,
    "v_mag": v_mag,
    "w_mag": w_mag,
    "u_hr": u_hr,
    "v_hr": v_hr,
    "w_hr": w_hr,
    "mask": mask,
}
if hr_mag is not None:
    tensor_map["hr_mag"] = hr_mag

for name, tensor in tensor_map.items():
    arr = tensor.detach().cpu().numpy().astype(np.float32)
    print(
        f"{name:6s} shape={tuple(arr.shape)} min={arr.min(): .4f} max={arr.max(): .4f} "
        f"mean={arr.mean(): .4f} std={arr.std(): .4f}"
    )

print("venc:", venc.detach().cpu().numpy())


In [ ]:
def center_slice(volume_3d: np.ndarray) -> np.ndarray:
    z = volume_3d.shape[-1] // 2
    return volume_3d[:, :, z]


sample = int(np.clip(SAMPLE_INDEX, 0, BATCH_SIZE - 1))

input_names = ["u", "v", "w", "u_mag", "v_mag", "w_mag"]
input_tensors = [u, v, w, u_mag, v_mag, w_mag]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, name, tensor in zip(axes.ravel(), input_names, input_tensors):
    vol = tensor[sample, 0].detach().cpu().numpy()
    slc = center_slice(vol)
    if "mag" in name:
        im = ax.imshow(slc, cmap="gray", vmin=0.0, vmax=1.0)
    else:
        im = ax.imshow(slc, cmap="coolwarm", vmin=-1.0, vmax=1.0)
    ax.set_title(name)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle(f"Model inputs (sample {sample}, center slice)")
plt.tight_layout()


In [ ]:
target_names = ["u_hr", "v_hr", "w_hr"]
target_tensors = [u_hr, v_hr, w_hr]
if hr_mag is not None:
    target_names.append("hr_mag")
    target_tensors.append(hr_mag)

fig, axes = plt.subplots(1, len(target_names), figsize=(4.8 * len(target_names), 4.2))
if len(target_names) == 1:
    axes = [axes]

for ax, name, tensor in zip(axes, target_names, target_tensors):
    vol = tensor[sample, 0].detach().cpu().numpy()
    slc = center_slice(vol)
    if "mag" in name:
        im = ax.imshow(slc, cmap="gray", vmin=0.0, vmax=1.0)
    else:
        im = ax.imshow(slc, cmap="coolwarm", vmin=-1.0, vmax=1.0)
    ax.set_title(name)
    ax.axis("off")
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle(f"Normalized HR targets (sample {sample})")
plt.tight_layout()


In [ ]:
mask_vol = mask[sample].detach().cpu().numpy().astype(np.float32)
z = mask_vol.shape[-1] // 2

hr_speed = torch.sqrt(u_hr[sample, 0] ** 2 + v_hr[sample, 0] ** 2 + w_hr[sample, 0] ** 2)
hr_speed = hr_speed.detach().cpu().numpy().astype(np.float32)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

im0 = axes[0].imshow(hr_speed[:, :, z], cmap="magma")
axes[0].set_title("|v| HR (center slice)")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

axes[1].imshow(hr_speed[:, :, z], cmap="gray")
im1 = axes[1].imshow(mask_vol[:, :, z], cmap="Reds", alpha=0.35, vmin=0.0, vmax=1.0)
axes[1].set_title("Mask overlay")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.tight_layout()


In [ ]:
N_BATCHES = 5
batch_means = {name: [] for name in ["u", "v", "w", "u_mag", "v_mag", "w_mag"]}

for i, b in zip(range(N_BATCHES), train_loader):
    u_b, v_b, w_b, u_mag_b, v_mag_b, w_mag_b = b[:6]
    for name, tensor in [
        ("u", u_b),
        ("v", v_b),
        ("w", w_b),
        ("u_mag", u_mag_b),
        ("v_mag", v_mag_b),
        ("w_mag", w_mag_b),
    ]:
        batch_means[name].append(float(tensor.mean().item()))

plt.figure(figsize=(10, 4))
for name, values in batch_means.items():
    plt.plot(range(1, len(values) + 1), values, marker="o", label=name)

plt.axhline(0.0, color="black", linewidth=1)
plt.xticks(range(1, N_BATCHES + 1))
plt.xlabel("Batch")
plt.ylabel("Tensor mean")
plt.title("Mean value drift across batches (quick sanity check)")
plt.legend(ncol=3)
plt.tight_layout()


## Model Inference Check (4 channels)

This section loads a trained checkpoint and runs a forward pass on the same batch to validate output shape and quality.


In [ ]:
from Network.SR4DFlowNet import SR4DFlowNet

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_CHECKPOINT = PROJECT_ROOT / "models" / "path_to_your_trained_model.pt"
LOW_RESBLOCK = 8
HI_RESBLOCK = 4
FORCE_PREDICT_MAG = True


def load_sr_model_from_checkpoint(
    checkpoint_path: Path,
    device: torch.device,
    res_increase: int,
    low_resblock: int,
    hi_resblock: int,
    force_predict_mag: bool = True,
):
    checkpoint = torch.load(str(checkpoint_path), map_location=device)
    predict_mag_flag = bool(checkpoint.get("predict_mag", force_predict_mag)) if isinstance(checkpoint, dict) else bool(force_predict_mag)

    if force_predict_mag and not predict_mag_flag:
        raise ValueError(
            "Checkpoint reports predict_mag=False, but this notebook expects 4-channel output."
        )

    model = SR4DFlowNet(
        res_increase=res_increase,
        low_resblock=low_resblock,
        hi_resblock=hi_resblock,
        predict_mag=predict_mag_flag,
    ).to(device)

    state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    return model, predict_mag_flag


if not MODEL_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {MODEL_CHECKPOINT}\n"
        "Update MODEL_CHECKPOINT to your trained .pt file."
    )

model, model_predict_mag = load_sr_model_from_checkpoint(
    checkpoint_path=MODEL_CHECKPOINT,
    device=DEVICE,
    res_increase=RES_INCREASE,
    low_resblock=LOW_RESBLOCK,
    hi_resblock=HI_RESBLOCK,
    force_predict_mag=FORCE_PREDICT_MAG,
)

print(f"Device: {DEVICE}")
print(f"Checkpoint: {MODEL_CHECKPOINT}")
print(f"model_predict_mag={model_predict_mag}")


In [ ]:
with torch.no_grad():
    pred = model(
        u.to(DEVICE),
        v.to(DEVICE),
        w.to(DEVICE),
        u_mag.to(DEVICE),
        v_mag.to(DEVICE),
        w_mag.to(DEVICE),
    ).detach().cpu()

print(f"Prediction shape: {tuple(pred.shape)}")
assert pred.ndim == 5, f"Expected 5D tensor [B,C,X,Y,Z], got {pred.ndim}D"
assert pred.shape[1] == 4, f"Expected 4 output channels (u,v,w,mag), got {pred.shape[1]}"

pred_u, pred_v, pred_w, pred_mag = pred[:, 0:1], pred[:, 1:2], pred[:, 2:3], pred[:, 3:4]

if hr_mag is None:
    raise ValueError("Ground-truth hr_mag is missing but 4-channel verification requires it.")

target_4ch = torch.cat([u_hr, v_hr, w_hr, hr_mag], dim=1).detach().cpu()
mse_per_channel = ((pred - target_4ch) ** 2).mean(dim=(0, 2, 3, 4)).numpy()
for name, mse in zip(["u", "v", "w", "mag"], mse_per_channel):
    print(f"MSE[{name}] = {mse:.6f}")


In [ ]:
channel_titles = ["u", "v", "w", "mag"]
pred_channels = [pred_u, pred_v, pred_w, pred_mag]
gt_channels = [u_hr, v_hr, w_hr, hr_mag]

fig, axes = plt.subplots(3, 4, figsize=(16, 11))
for col, (title, pred_ch, gt_ch) in enumerate(zip(channel_titles, pred_channels, gt_channels)):
    pred_vol = pred_ch[sample, 0].numpy()
    gt_vol = gt_ch[sample, 0].detach().cpu().numpy()
    err_vol = np.abs(pred_vol - gt_vol)

    pred_slc = center_slice(pred_vol)
    gt_slc = center_slice(gt_vol)
    err_slc = center_slice(err_vol)

    if title == "mag":
        pred_im = axes[0, col].imshow(pred_slc, cmap="gray", vmin=0.0, vmax=1.0)
        gt_im = axes[1, col].imshow(gt_slc, cmap="gray", vmin=0.0, vmax=1.0)
    else:
        pred_im = axes[0, col].imshow(pred_slc, cmap="coolwarm", vmin=-1.0, vmax=1.0)
        gt_im = axes[1, col].imshow(gt_slc, cmap="coolwarm", vmin=-1.0, vmax=1.0)

    err_im = axes[2, col].imshow(err_slc, cmap="magma")

    axes[0, col].set_title(f"Pred {title}")
    axes[1, col].set_title(f"GT {title}")
    axes[2, col].set_title(f"|Err| {title}")

    for row in range(3):
        axes[row, col].axis("off")

    plt.colorbar(pred_im, ax=axes[0, col], fraction=0.046)
    plt.colorbar(gt_im, ax=axes[1, col], fraction=0.046)
    plt.colorbar(err_im, ax=axes[2, col], fraction=0.046)

plt.suptitle(f"Prediction vs target (sample {sample}, center slice)")
plt.tight_layout()


## Quick interpretation

- If velocity channels look saturated or shifted, re-check `RAW_PHASE_INPUT`, `venc`, and `LEGACY_INVERT_UV_SIGN_ON_RAW`.
- If mask alignment is off, verify HR/mask registration and CSV paths.
- For deterministic debugging, set `NO_AUGMENTATION=True` and `DETERMINISTIC_TRAIN_PATCHES=True`.
- For strict 4-channel checks, keep `PREDICT_MAG=True` and `FORCE_PREDICT_MAG=True`.
